# Refresh semantic models

Last step of the `Refresh_Orchestration` pipeline. Refreshes the Direct Lake semantic model so the Power BI report shows
the newly loaded Gold data. Models that are not deployed yet (first run of a new installation) are skipped.

In [ ]:
semantic_models = "StravaInsights_SM"
timeout_minutes = 30
gold_lakehouse = "Strava_Gold"

In [ ]:
import json
import time
import sempy.fabric as fabric

workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
client = fabric.PowerBIRestClient()
results = {}

# Direct Lake reads the Gold tables through the lakehouse SQL endpoint. Sync its metadata first, so tables created or
# changed by this run (e.g. on the first run of a new installation) are visible to the model refresh.
endpoints = fabric.list_items(type="SQLEndpoint")
gold_endpoint = endpoints.loc[endpoints["Display Name"] == gold_lakehouse, "Id"]
if len(gold_endpoint):
    rest = fabric.FabricRestClient()
    response = rest.post(f"v1/workspaces/{workspace_id}/sqlEndpoints/{gold_endpoint.iloc[0]}/refreshMetadata", json={})
    location = response.headers.get("Location")
    deadline = time.time() + 10 * 60
    while response.status_code == 202 and location and time.time() < deadline:
        time.sleep(int(response.headers.get("Retry-After", 5)))
        response = rest.get(location)
        if response.json().get("status") in ("Succeeded", "Failed", "Cancelled"):
            break
    print(f"SQL endpoint metadata sync: HTTP {response.status_code} {response.json().get('status', '') if response.content else ''}")

deployed = set(fabric.list_items(type="SemanticModel")["Display Name"])
for model_name in [m.strip() for m in semantic_models.split(",") if m.strip()]:
    if model_name not in deployed:
        results[model_name] = "SkippedNotDeployed"
        print(f"{model_name}: not deployed in this workspace yet - skipped")
        continue
    dataset_id = fabric.resolve_dataset_id(model_name)
    base = f"v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/refreshes"
    response = client.post(base, json={"type": "full", "retryCount": 1})
    if response.status_code not in (200, 202):
        raise RuntimeError(f"Refresh of {model_name} could not start: HTTP {response.status_code} {response.text[:300]}")
    deadline = time.time() + timeout_minutes * 60
    status = "Unknown"
    while time.time() < deadline:
        time.sleep(10)
        latest = client.get(f"{base}?$top=1").json()["value"][0]
        status = latest.get("status", "Unknown")
        if status != "Unknown":
            break
    results[model_name] = status
    print(f"{model_name}: {status}")
    if status != "Completed":
        raise RuntimeError(f"Refresh of {model_name} ended with status {status}: {latest.get('serviceExceptionJson')}")

notebookutils.notebook.exit(json.dumps({"status": "success", "refreshed": results}))